# Temporal Robustness Experiments

This notebook investigates and addresses the gap between validation ROC-AUC (`0.7072`) and held-out time-proxy test ROC-AUC (`0.6607`). It keeps the final 15% of users completely untouched during feature-set, weighting, and regularization selection.

The experiment uses two rolling development folds, compares row-level and latest-customer ranking quality, diagnoses feature drift using development data only, and promotes a replacement model only when the untouched test improves without sacrificing PR-AUC.


## 1. Imports and experiment contract

Load pinned dependencies and define canonical artifact paths.

In [1]:
from __future__ import annotations

import gc
import json
import math
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    precision_recall_fscore_support,
    roc_auc_score,
)
from xgboost import XGBClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FEATURE_MATRIX = ROOT / "data" / "processed" / "instacart" / "features" / "instacart_feature_matrix.pkl"
LABELS = ROOT / "data" / "processed" / "instacart" / "labels" / "instacart_phase2_decay_labels.csv"
PREDICTIONS_DIR = ROOT / "data" / "processed" / "instacart" / "predictions"
MODEL_DIR = ROOT / "models" / "phase4"
REPORT_DIR = ROOT / "reports" / "modeling" / "instacart"
PLOTS_DIR = REPORT_DIR / "plots"
TARGET = "early_decay_label"
KEYS = ["user_id", "order_id"]
PROXY_SPLIT = "leading_time_proxy_split"
BASELINE_REPORT = REPORT_DIR / "phase4_leading_xgboost_report.json"
BASELINE_PREDICTIONS = PREDICTIONS_DIR / "leading_xgboost_time_proxy_test_predictions.csv"
RANDOM_STATE = 42


## 2. Leakage-safe evaluation helpers

Create the untouched test boundary, development-only drift diagnostics, user-balanced weights, ranking metrics, and model helpers.

In [2]:
def top_lift(y_true: pd.Series, scores: np.ndarray, fraction: float = 0.10) -> float:
    ranked = pd.DataFrame({"target": y_true.astype(int).to_numpy(), "score": scores})
    count = max(1, int(len(ranked) * fraction))
    top_rate = float(ranked.nlargest(count, "score")["target"].mean())
    base_rate = float(ranked["target"].mean())
    return float(top_rate / base_rate) if base_rate else float("nan")


def ranking_metrics(frame: pd.DataFrame, scores: np.ndarray) -> dict[str, float]:
    scored = frame[[*KEYS, "relative_day", TARGET]].copy()
    scored["score"] = scores
    y = scored[TARGET].astype(bool)
    latest = (
        scored.sort_values(["user_id", "relative_day", "order_id"], kind="mergesort")
        .groupby("user_id", as_index=False)
        .tail(1)
    )
    latest_y = latest[TARGET].astype(bool)
    return {
        "row_positive_rate": float(y.mean()),
        "row_roc_auc": float(roc_auc_score(y, scores)),
        "row_pr_auc": float(average_precision_score(y, scores)),
        "row_top_10_lift": top_lift(y, scores),
        "latest_positive_rate": float(latest_y.mean()),
        "latest_roc_auc": float(roc_auc_score(latest_y, latest["score"])),
        "latest_pr_auc": float(average_precision_score(latest_y, latest["score"])),
        "latest_top_10_lift": top_lift(latest_y, latest["score"].to_numpy()),
    }


def evaluation_metrics(frame: pd.DataFrame, scores: np.ndarray, threshold: float) -> dict[str, object]:
    y = frame[TARGET].astype(bool)
    predictions = scores >= threshold
    precision, recall, f1, _ = precision_recall_fscore_support(
        y, predictions, average="binary", zero_division=0
    )
    return {
        "roc_auc": float(roc_auc_score(y, scores)),
        "pr_auc": float(average_precision_score(y, scores)),
        "threshold": float(threshold),
        "precision_at_threshold": float(precision),
        "recall_at_threshold": float(recall),
        "f1_at_threshold": float(f1),
        "top_5_pct": {
            "fraction": 0.05,
            "rows": max(1, int(len(y) * 0.05)),
            "lift": top_lift(y, scores, 0.05),
        },
        "top_10_pct": {
            "fraction": 0.10,
            "rows": max(1, int(len(y) * 0.10)),
            "lift": top_lift(y, scores, 0.10),
        },
    }


def load_modeling_data() -> tuple[pd.DataFrame, list[str], list[str]]:
    matrix = pd.read_pickle(FEATURE_MATRIX)
    matrix[TARGET] = matrix[TARGET].astype(bool)
    labels = pd.read_csv(
        LABELS,
        usecols=[
            "user_id",
            "order_id",
            "relative_day",
            "next_gap_days",
            "historical_median_gap_days",
        ],
    )
    modeling = matrix.merge(labels, on=KEYS, how="left", validate="one_to_one")
    completion = (
        modeling.groupby("user_id", as_index=False)["relative_day"]
        .max()
        .rename(columns={"relative_day": "user_observation_completion_day"})
        .sort_values(["user_observation_completion_day", "user_id"], kind="mergesort")
        .reset_index(drop=True)
    )
    # One-based empirical rank reproduces the original floor-index split boundaries exactly.
    completion["user_rank_fraction"] = (np.arange(len(completion)) + 1.0) / len(completion)
    completion[PROXY_SPLIT] = "test"
    completion.loc[completion["user_rank_fraction"] < 0.70, PROXY_SPLIT] = "train"
    completion.loc[
        completion["user_rank_fraction"].between(0.70, 0.85, inclusive="left"),
        PROXY_SPLIT,
    ] = "validation"
    modeling = modeling.merge(completion, on="user_id", how="left", validate="many_to_one")
    non_features = {*KEYS, "split", TARGET}
    combined = [column for column in matrix.columns if column not in non_features]
    leading = [column for column in combined if not column.startswith("base_")]
    return modeling, leading, combined


def standardized_drift(train: pd.DataFrame, validation: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []
    for feature in features:
        left = pd.to_numeric(train[feature], errors="coerce")
        right = pd.to_numeric(validation[feature], errors="coerce")
        pooled = np.sqrt((left.var() + right.var()) / 2)
        smd = (
            abs(float(right.mean() - left.mean()) / float(pooled))
            if pooled and np.isfinite(pooled)
            else 0.0
        )
        rows.append(
            {
                "feature": feature,
                "abs_standardized_mean_difference": smd,
                "train_mean": float(left.mean()),
                "validation_mean": float(right.mean()),
            }
        )
    return pd.DataFrame(rows).sort_values("abs_standardized_mean_difference", ascending=False)


def feature_sets(
    leading: list[str], combined: list[str], development_drift: pd.DataFrame
) -> dict[str, list[str]]:
    stable = development_drift.loc[
        development_drift["abs_standardized_mean_difference"] <= 0.45, "feature"
    ].tolist()
    mandatory = {
        "current_gap_ratio_to_historical_median",
        "gap_slope_last3",
        "purchase_frequency_slope_last3",
        "latest_gap_ratio_to_prior_avg",
    }
    stable = [feature for feature in combined if feature in set(stable) | mandatory]
    change_tokens = ("ratio", "delta", "slope", "flag", "distance")
    change = [feature for feature in leading if any(token in feature for token in change_tokens)]
    for feature in sorted(mandatory):
        if feature in leading and feature not in change:
            change.append(feature)
    return {
        "leading": leading,
        "combined": combined,
        "drift_robust": stable,
        "change_only": change,
    }


def sample_weights(frame: pd.DataFrame, mode: str) -> np.ndarray:
    weights = np.ones(len(frame), dtype="float64")
    if "user" in mode:
        observations = frame.groupby("user_id")["user_id"].transform("size").to_numpy()
        weights /= observations
    if "recency" in mode:
        rank = frame["user_rank_fraction"].to_numpy(dtype="float64")
        local_min, local_max = float(rank.min()), float(rank.max())
        normalized = (rank - local_min) / max(local_max - local_min, 1e-9)
        weights *= 0.50 + 1.50 * normalized
    weights /= weights.mean()
    return weights.astype("float32")


PARAMETER_PROFILES = {
    "current": {
        "n_estimators": 320,
        "max_depth": 6,
        "learning_rate": 0.06,
        "subsample": 0.85,
        "colsample_bytree": 0.90,
        "min_child_weight": 3,
        "reg_lambda": 2.0,
        "reg_alpha": 0.0,
        "gamma": 0.0,
    },
    "regularized": {
        "n_estimators": 600,
        "max_depth": 4,
        "learning_rate": 0.035,
        "subsample": 0.85,
        "colsample_bytree": 0.78,
        "min_child_weight": 15,
        "reg_lambda": 8.0,
        "reg_alpha": 0.50,
        "gamma": 0.05,
    },
}


def fit_model(
    train: pd.DataFrame,
    validation: pd.DataFrame | None,
    features: list[str],
    weight_mode: str,
    profile: str,
    n_estimators: int | None = None,
) -> XGBClassifier:
    weights = sample_weights(train, weight_mode)
    y_train = train[TARGET].astype(bool)
    negative_weight = float(weights[~y_train.to_numpy()].sum())
    positive_weight = float(weights[y_train.to_numpy()].sum())
    params = PARAMETER_PROFILES[profile].copy()
    if n_estimators is not None:
        params["n_estimators"] = int(n_estimators)
    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        scale_pos_weight=(negative_weight / max(positive_weight, 1e-9)) * 0.65,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=30 if validation is not None else None,
        **params,
    )
    fit_kwargs: dict[str, object] = {
        "X": train[features],
        "y": y_train,
        "sample_weight": weights,
        "verbose": False,
    }
    if validation is not None:
        fit_kwargs["eval_set"] = [(validation[features], validation[TARGET].astype(bool))]
    model.fit(**fit_kwargs)
    return model


def timing_metric(test_frame: pd.DataFrame, threshold: float) -> dict[str, object]:
    scored = test_frame.copy()
    scored["flagged"] = scored["risk_score"] >= threshold
    positive_events = scored[scored[TARGET]].copy()
    positive_events["decay_event_day"] = (
        positive_events["relative_day"] + 2.0 * positive_events["historical_median_gap_days"]
    )
    positive_events["next_observed_order_day"] = (
        positive_events["relative_day"] + positive_events["next_gap_days"]
    )
    first_event = (
        positive_events.sort_values(["user_id", "decay_event_day", "order_id"])
        .groupby("user_id", as_index=False)
        .first()
    )
    candidates = scored[scored["flagged"]].merge(
        first_event[["user_id", "decay_event_day", "next_observed_order_day"]],
        on="user_id",
        how="inner",
        validate="many_to_one",
    )
    candidates = candidates[candidates["relative_day"] <= candidates["decay_event_day"]]
    first_flag = (
        candidates.sort_values(["user_id", "relative_day", "order_id"])
        .groupby("user_id", as_index=False)
        .first()
    )
    first_flag["days_before_decay_threshold"] = (
        first_flag["decay_event_day"] - first_flag["relative_day"]
    )
    first_flag["days_before_next_observed_order"] = (
        first_flag["next_observed_order_day"] - first_flag["relative_day"]
    )

    def summarize(series: pd.Series) -> dict[str, float]:
        return {
            "mean": float(series.mean()),
            "median": float(series.median()),
            "p25": float(series.quantile(0.25)),
            "p75": float(series.quantile(0.75)),
            "min": float(series.min()),
            "max": float(series.max()),
        }

    return {
        "positive_event_users": int(first_event["user_id"].nunique()),
        "correctly_flagged_users": int(first_flag["user_id"].nunique()),
        "correctly_flagged_user_rate": float(
            first_flag["user_id"].nunique() / max(first_event["user_id"].nunique(), 1)
        ),
        "threshold_definition": "rolling-development top-10% score cutoff",
        "event_definition": "relative_day + 2 * historical_median_gap_days",
        "days_before_decay_threshold_summary": summarize(first_flag["days_before_decay_threshold"]),
        "days_before_next_observed_order_summary": summarize(
            first_flag["days_before_next_observed_order"]
        ),
    }


## 3. Rolling development experiments

Compare candidate feature sets and weighting strategies on two lifecycle-ordered development folds. No candidate reads the final test cohort.

In [3]:
CANDIDATES = [
    {"name": "current_leading_row", "feature_set": "leading", "weights": "row", "profile": "current"},
    {"name": "regularized_leading_row", "feature_set": "leading", "weights": "row", "profile": "regularized"},
    {"name": "regularized_leading_user", "feature_set": "leading", "weights": "user", "profile": "regularized"},
    {"name": "regularized_combined_row", "feature_set": "combined", "weights": "row", "profile": "regularized"},
    {"name": "regularized_combined_user", "feature_set": "combined", "weights": "user", "profile": "regularized"},
    {"name": "regularized_combined_user_recency", "feature_set": "combined", "weights": "user_recency", "profile": "regularized"},
    {"name": "regularized_drift_robust_user_recency", "feature_set": "drift_robust", "weights": "user_recency", "profile": "regularized"},
    {"name": "regularized_change_only_user_recency", "feature_set": "change_only", "weights": "user_recency", "profile": "regularized"},
]


def run_development_experiments(
    modeling: pd.DataFrame, sets: dict[str, list[str]]
) -> tuple[pd.DataFrame, dict[str, object]]:
    folds = [
        ("fold_1", 0.55, 0.55, 0.70),
        ("fold_2", 0.70, 0.70, 0.85),
    ]
    results: list[dict[str, object]] = []
    for candidate in CANDIDATES:
        features = sets[candidate["feature_set"]]
        for fold_name, train_end, validation_start, validation_end in folds:
            train = modeling[modeling["user_rank_fraction"] < train_end]
            validation = modeling[
                modeling["user_rank_fraction"].between(
                    validation_start, validation_end, inclusive="left"
                )
            ]
            started = time.perf_counter()
            model = fit_model(
                train,
                validation,
                features,
                candidate["weights"],
                candidate["profile"],
            )
            scores = model.predict_proba(validation[features])[:, 1]
            metrics = ranking_metrics(validation, scores)
            result = {
                **candidate,
                "fold": fold_name,
                "feature_count": len(features),
                "best_iteration": int(model.best_iteration),
                "top_10_threshold": float(np.quantile(scores, 0.90)),
                "seconds": round(time.perf_counter() - started, 2),
                **metrics,
            }
            results.append(result)
            print(
                f"{candidate['name']} {fold_name}: "
                f"ROC={metrics['row_roc_auc']:.4f} PR={metrics['row_pr_auc']:.4f} "
                f"latest_ROC={metrics['latest_roc_auc']:.4f} latest_PR={metrics['latest_pr_auc']:.4f}"
            )
            del model, scores
            gc.collect()

    detailed = pd.DataFrame(results)
    summary = (
        detailed.groupby(["name", "feature_set", "weights", "profile", "feature_count"], as_index=False)
        .agg(
            mean_row_roc_auc=("row_roc_auc", "mean"),
            mean_row_pr_auc=("row_pr_auc", "mean"),
            mean_row_top_10_lift=("row_top_10_lift", "mean"),
            mean_latest_roc_auc=("latest_roc_auc", "mean"),
            mean_latest_pr_auc=("latest_pr_auc", "mean"),
            mean_latest_top_10_lift=("latest_top_10_lift", "mean"),
            mean_best_iteration=("best_iteration", "mean"),
        )
    )
    summary["row_pr_lift"] = summary["mean_row_pr_auc"] / detailed.groupby("name")[
        "row_positive_rate"
    ].mean().reindex(summary["name"]).to_numpy()
    summary["latest_pr_lift"] = summary["mean_latest_pr_auc"] / detailed.groupby("name")[
        "latest_positive_rate"
    ].mean().reindex(summary["name"]).to_numpy()
    summary["robustness_score"] = (
        0.45 * summary["row_pr_lift"]
        + 0.35 * summary["latest_pr_lift"]
        + 0.10 * summary["mean_row_roc_auc"]
        + 0.10 * summary["mean_latest_roc_auc"]
    )
    summary = summary.sort_values(
        ["robustness_score", "mean_row_pr_auc", "mean_row_roc_auc"], ascending=False
    ).reset_index(drop=True)
    selected = summary.iloc[0].to_dict()
    selected["threshold"] = float(
        detailed.loc[
            detailed["name"].eq(selected["name"]) & detailed["fold"].eq("fold_2"),
            "top_10_threshold",
        ].iloc[0]
    )
    return detailed, {"summary": summary, "selected": selected}


## 4. Final selection and guarded promotion

Select using development folds, fit once on the first 85% of users, evaluate the untouched 15%, and replace canonical artifacts only if the guarded improvement criteria pass.

In [4]:
def save_shap_and_plots(
    model: XGBClassifier,
    X_test: pd.DataFrame,
    keys: pd.DataFrame,
    output_path: Path,
) -> None:
    contributions = model.get_booster().predict(
        xgb.DMatrix(X_test, feature_names=X_test.columns.tolist()), pred_contribs=True
    )
    shap_columns = [f"shap_{feature}" for feature in X_test.columns]
    shap_frame = pd.DataFrame(contributions[:, :-1], columns=shap_columns, index=X_test.index)
    shap_frame.insert(0, "order_id", keys["order_id"].to_numpy())
    shap_frame.insert(0, "user_id", keys["user_id"].to_numpy())
    shap_frame["shap_bias"] = contributions[:, -1]
    shap_frame.to_pickle(output_path)

    values = pd.DataFrame(contributions[:, :-1], columns=X_test.columns)
    plot_specs = [
        (values.abs().mean().nlargest(15), "Mean absolute SHAP value", "Global SHAP feature importance", "shap_global_importance.png"),
        (values.clip(lower=0).mean().nlargest(15), "Mean positive SHAP value", "Strongest positive risk drivers", "shap_positive_drivers.png"),
        ((-values.clip(upper=0).mean()).nlargest(15), "Mean protective SHAP magnitude", "Strongest protective drivers", "shap_protective_drivers.png"),
    ]
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)
    for series, xlabel, title, filename in plot_specs:
        fig, ax = plt.subplots(figsize=(10, 6))
        series.sort_values().plot.barh(ax=ax, color="#2563eb")
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("")
        fig.tight_layout()
        fig.savefig(PLOTS_DIR / filename, dpi=150)
        plt.close(fig)


def write_experiment_report(
    summary: pd.DataFrame,
    selected: dict[str, object],
    baseline: dict[str, float],
    test_metrics: dict[str, object],
    latest_metrics: dict[str, float],
    promoted: bool,
    feature_count: int,
) -> None:
    lines = [
        "# Temporal Robustness Experiment",
        "",
        "## Decision",
        "",
        f"- Replacement promoted: **{'yes' if promoted else 'no'}**",
        f"- Selected development candidate: `{selected['name']}`",
        f"- Selected feature count: {feature_count}",
        "- Selection used two rolling development folds; the final 15% user cohort remained untouched until this decision.",
        "",
        "## Untouched Test Comparison",
        "",
        "| Metric | Previous model | Candidate | Change |",
        "|---|---:|---:|---:|",
        f"| Row ROC-AUC | {baseline['row_roc_auc']:.4f} | {test_metrics['roc_auc']:.4f} | {test_metrics['roc_auc'] - baseline['row_roc_auc']:+.4f} |",
        f"| Row PR-AUC | {baseline['row_pr_auc']:.4f} | {test_metrics['pr_auc']:.4f} | {test_metrics['pr_auc'] - baseline['row_pr_auc']:+.4f} |",
        f"| Latest-customer ROC-AUC | {baseline['latest_roc_auc']:.4f} | {latest_metrics['latest_roc_auc']:.4f} | {latest_metrics['latest_roc_auc'] - baseline['latest_roc_auc']:+.4f} |",
        f"| Latest-customer PR-AUC | {baseline['latest_pr_auc']:.4f} | {latest_metrics['latest_pr_auc']:.4f} | {latest_metrics['latest_pr_auc'] - baseline['latest_pr_auc']:+.4f} |",
        "",
        "## Rolling Development Results",
        "",
        "| Candidate | Features | Weighting | Mean ROC-AUC | Mean PR-AUC | Latest ROC-AUC | Latest PR-AUC |",
        "|---|---:|---|---:|---:|---:|---:|",
    ]
    for row in summary.itertuples():
        lines.append(
            f"| `{row.name}` | {int(row.feature_count)} | {row.weights} | "
            f"{row.mean_row_roc_auc:.4f} | {row.mean_row_pr_auc:.4f} | "
            f"{row.mean_latest_roc_auc:.4f} | {row.mean_latest_pr_auc:.4f} |"
        )
    lines.extend(
        [
            "",
            "## Interpretation",
            "",
            "The experiment targets lifecycle distribution shift rather than optimizing against the held-out test. User-balanced weighting makes each customer contribute equal total training weight; recency weighting emphasizes later development histories; drift-robust feature sets are selected using development folds only.",
        ]
    )
    (REPORT_DIR / "temporal_robustness_report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def main() -> dict[str, object]:
    PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    modeling, leading, combined = load_modeling_data()
    development_train = modeling[modeling["user_rank_fraction"] < 0.70]
    development_validation = modeling[
        modeling["user_rank_fraction"].between(0.70, 0.85, inclusive="left")
    ]
    drift = standardized_drift(development_train, development_validation, combined)
    sets = feature_sets(leading, combined, drift)
    print("Feature sets:", {name: len(columns) for name, columns in sets.items()})
    display(drift.head(15))

    detailed, selection = run_development_experiments(modeling, sets)
    summary = selection["summary"]
    selected = selection["selected"]
    display(summary)

    selected_features = sets[str(selected["feature_set"])]
    final_train = modeling[modeling["user_rank_fraction"] < 0.85]
    untouched_test = modeling[modeling["user_rank_fraction"] >= 0.85].copy()
    estimator_count = max(80, int(math.ceil(float(selected["mean_best_iteration"]) + 1)))
    final_model = fit_model(
        final_train,
        None,
        selected_features,
        str(selected["weights"]),
        str(selected["profile"]),
        n_estimators=estimator_count,
    )
    test_scores = final_model.predict_proba(untouched_test[selected_features])[:, 1]
    threshold = float(selected["threshold"])
    test_metrics = evaluation_metrics(untouched_test, test_scores, threshold)
    latest_metrics = ranking_metrics(untouched_test, test_scores)

    # Frozen pre-experiment metrics prevent a rerun from comparing against artifacts it just promoted.
    baseline = {
        "row_roc_auc": 0.660650352675686,
        "row_pr_auc": 0.25852226020555197,
        "latest_roc_auc": 0.6603807196592272,
        "latest_pr_auc": 0.2650062454050235,
    }
    promoted = bool(
        test_metrics["roc_auc"] > baseline["row_roc_auc"] + 0.001
        and test_metrics["pr_auc"] >= baseline["row_pr_auc"] - 0.002
        and latest_metrics["latest_roc_auc"] > baseline["latest_roc_auc"]
    )

    test_predictions = untouched_test[
        [*KEYS, PROXY_SPLIT, TARGET, "relative_day", "next_gap_days", "historical_median_gap_days"]
    ].copy()
    test_predictions["risk_score"] = test_scores
    test_predictions["flagged_at_threshold"] = test_scores >= threshold
    timing = timing_metric(test_predictions, threshold)

    write_experiment_report(
        summary,
        selected,
        baseline,
        test_metrics,
        latest_metrics,
        promoted,
        len(selected_features),
    )
    experiment_payload = {
        "selected_candidate": selected,
        "selected_features": selected_features,
        "development_results": detailed.to_dict(orient="records"),
        "baseline_test_metrics": baseline,
        "candidate_test_metrics": test_metrics,
        "candidate_latest_metrics": latest_metrics,
        "promoted": promoted,
    }
    (REPORT_DIR / "temporal_robustness_report.json").write_text(
        json.dumps(experiment_payload, indent=2, default=str) + "\n", encoding="utf-8"
    )

    if promoted:
        predictions_path = PREDICTIONS_DIR / "leading_xgboost_time_proxy_test_predictions.csv"
        shap_path = PREDICTIONS_DIR / "leading_xgboost_time_proxy_test_shap_values.pkl"
        model_path = MODEL_DIR / "leading_xgboost_time_proxy.pkl"
        test_predictions.to_csv(predictions_path, index=False)
        joblib.dump(
            {
                "model": final_model,
                "feature_columns": selected_features,
                "threshold": threshold,
                "split_method": "user_level_relative_time_proxy_rolling_robustness",
                "selected_candidate": selected,
            },
            model_path,
        )
        save_shap_and_plots(
            final_model,
            untouched_test[selected_features],
            untouched_test[KEYS],
            shap_path,
        )
        split_summary = (
            modeling.groupby(PROXY_SPLIT)
            .agg(
                rows=(TARGET, "count"),
                users=("user_id", "nunique"),
                positives=(TARGET, "sum"),
                positive_rate=(TARGET, "mean"),
                min_completion_day=("user_observation_completion_day", "min"),
                max_completion_day=("user_observation_completion_day", "max"),
            )
            .reset_index()
        )
        canonical_report = {
            "matrix_rows": int(len(modeling)),
            "leading_feature_count": int(len(selected_features)),
            "leading_feature_columns": selected_features,
            "split_method": "user_level_relative_time_proxy_with_rolling_development_selection",
            "split_caveat": "Instacart has no global calendar dates; relative_day is user-lifecycle time, not real calendar time.",
            "split_summary": split_summary.to_dict(orient="records"),
            "best_validation_result": {"params": selected},
            "test_metrics": test_metrics,
            "latest_customer_metrics": latest_metrics,
            "timing_metric": timing,
            "outputs": {
                "trained_model": model_path.relative_to(ROOT).as_posix(),
                "test_predictions": predictions_path.relative_to(ROOT).as_posix(),
                "test_shap_values": shap_path.relative_to(ROOT).as_posix(),
                "report_md": "reports/modeling/instacart/phase4_leading_xgboost_report.md",
                "report_json": "reports/modeling/instacart/phase4_leading_xgboost_report.json",
            },
        }
        BASELINE_REPORT.write_text(
            json.dumps(canonical_report, indent=2, default=str) + "\n", encoding="utf-8"
        )
        report_lines = [
            "# Phase 4 Temporally Robust XGBoost Report",
            "",
            "## Scope",
            "",
            "Promoted model selected across two rolling user-lifecycle development folds, then evaluated once on the untouched final 15% user cohort.",
            "",
            "## Test Metrics",
            "",
            f"- ROC-AUC: {test_metrics['roc_auc']:.4f}",
            f"- PR-AUC: {test_metrics['pr_auc']:.4f}",
            f"- Top 5% lift: {test_metrics['top_5_pct']['lift']:.2f}x",
            f"- Top 10% lift: {test_metrics['top_10_pct']['lift']:.2f}x",
            f"- Latest-customer ROC-AUC: {latest_metrics['latest_roc_auc']:.4f}",
            f"- Latest-customer PR-AUC: {latest_metrics['latest_pr_auc']:.4f}",
            f"- Precision at rolling-development threshold: {test_metrics['precision_at_threshold']:.2%}",
            f"- Recall at rolling-development threshold: {test_metrics['recall_at_threshold']:.2%}",
            "",
            "## Timing Metric",
            "",
            f"- Correctly flagged users: {timing['correctly_flagged_users']:,} / {timing['positive_event_users']:,}",
            f"- Correctly flagged user rate: {timing['correctly_flagged_user_rate']:.2%}",
            f"- Median lead time: {timing['days_before_decay_threshold_summary']['median']:.1f} days",
            "",
            "## Selected Design",
            "",
            f"- Candidate: `{selected['name']}`",
            f"- Features: {len(selected_features)}",
            f"- Weighting: `{selected['weights']}`",
            f"- Regularization profile: `{selected['profile']}`",
            "",
            "See `reports/modeling/instacart/temporal_robustness_report.md` for all development candidates and the honest pre/post comparison.",
        ]
        (REPORT_DIR / "phase4_leading_xgboost_report.md").write_text(
            "\n".join(report_lines) + "\n", encoding="utf-8"
        )

    result = {
        "selected": selected,
        "baseline_test": baseline,
        "candidate_test": test_metrics,
        "candidate_latest": latest_metrics,
        "promoted": promoted,
        "feature_count": len(selected_features),
    }
    print(json.dumps(result, indent=2, default=str))
    return result


## 5. Execute

Run the full experiment and retain all candidate and final outputs.

In [5]:
from IPython.display import display

result = main()


Feature sets: {'leading': 38, 'combined': 45, 'drift_robust': 38, 'change_only': 23}


,feature,abs_standardized_mean_difference,train_mean,validation_mean
38,base_user_tenure_days,0.940094,92.484964,153.738186
37,history_reliability_score,0.663143,0.879503,0.950611
34,behavior_orders_to_date,0.577143,12.162792,20.682675
35,known_gap_count_feature,0.577143,11.162792,19.682675
39,base_total_orders_to_date,0.577143,12.162792,20.682675
17,reorder_ratio_prior_to_current_avg,0.555762,0.423778,0.533541
42,base_avg_reorder_ratio_to_date,0.531174,0.449823,0.549234
16,reorder_ratio_recent3_avg,0.423947,0.580605,0.678861
36,trend_history_available,0.319407,0.871053,0.959002
18,reorder_ratio_recent3_ratio_to_prior,0.246819,1.521800,1.363542


current_leading_row fold_1: ROC=0.7403 PR=0.3934 latest_ROC=0.8336 latest_PR=0.5366


current_leading_row fold_2: ROC=0.7070 PR=0.3478 latest_ROC=0.7601 latest_PR=0.4693


regularized_leading_row fold_1: ROC=0.7403 PR=0.3929 latest_ROC=0.8337 latest_PR=0.5374


...

regularized_drift_robust_user_recency fold_2: ROC=0.7053 PR=0.3442 latest_ROC=0.7598 latest_PR=0.4659


regularized_change_only_user_recency fold_1: ROC=0.7190 PR=0.3703 latest_ROC=0.8136 latest_PR=0.5223


regularized_change_only_user_recency fold_2: ROC=0.6929 PR=0.3335 latest_ROC=0.7478 latest_PR=0.4648


,name,feature_set,weights,profile,feature_count,mean_row_roc_auc,mean_row_pr_auc,mean_row_top_10_lift,mean_latest_roc_auc,mean_latest_pr_auc,mean_latest_top_10_lift,mean_best_iteration,row_pr_lift,latest_pr_lift,robustness_score
0,current_leading_row,leading,row,current,38,0.723689,0.370614,2.253341,0.796826,0.502959,2.406021,182.0,1.908012,2.089130,1.741852
1,regularized_leading_row,leading,row,regularized,38,0.723610,0.370157,2.246669,0.796500,0.502322,2.394791,593.0,1.905660,2.086483,1.739827
2,regularized_combined_row,combined,row,regularized,45,0.720090,0.367289,2.226998,0.795706,0.499156,2.356013,351.5,1.890894,2.073333,1.728149
3,regularized_drift_robust_user_recency,drift_robust,user_recency,regularized,38,0.721089,0.365913,2.208602,0.796566,0.501212,2.377864,582.0,1.883813,2.081872,1.728136
4,regularized_combined_user_recency,combined,user_recency,regularized,45,0.717791,0.362885,2.185740,0.796267,0.502150,2.387606,567.0,1.868225,2.085767,1.722125
5,regularized_leading_user,leading,user,regularized,38,0.718806,0.361126,2.163298,0.795936,0.500907,2.353520,595.5,1.859167,2.080604,1.716311
6,regularized_combined_user,combined,user,regularized,45,0.717656,0.361319,2.175352,0.796030,0.500640,2.380055,499.0,1.860161,2.079498,1.716265
7,regularized_change_only_user_recency,change_only,user_recency,regularized,23,0.705950,0.351922,2.153581,0.780695,0.493558,2.407667,599.0,1.811781,2.050079,1.681493


{
  "selected": {
    "name": "current_leading_row",
    "feature_set": "leading",
    "weights": "row",
    "profile": "current",
    "feature_count": 38,
    "mean_row_roc_auc": 0.7236889793861694,
    "mean_row_pr_auc": 0.37061365070173635,
    "mean_row_top_10_lift": 2.253341239646259,
    "mean_latest_roc_auc": 0.7968259583717252,
    "mean_latest_pr_auc": 0.5029593607231833,
    "mean_latest_top_10_lift": 2.406020735918528,
    "mean_best_iteration": 182.0,
    "row_pr_lift": 1.9080116028244933,
    "latest_pr_lift": 2.0891297512788594,
    "robustness_score": 1.7418521279944124,
    "threshold": 0.5835505723953247
  },
  "baseline_test": {
    "row_roc_auc": 0.660650352675686,
    "row_pr_auc": 0.25852226020555197,
    "latest_roc_auc": 0.6603807196592272,
    "latest_pr_auc": 0.2650062454050235
  },
  "candidate_test": {
    "roc_auc": 0.6640226373862553,
    "pr_auc": 0.2636898077924394,
    "threshold": 0.5835505723953247,
    "precision_at_threshold": 0.3600984975791558,
   

## 6. Results

Render the full development comparison and the promoted model report, if promotion was justified.

In [6]:
from IPython.display import Markdown, display

display(Markdown((REPORT_DIR / "temporal_robustness_report.md").read_text(encoding="utf-8")))
if result["promoted"]:
    display(Markdown((REPORT_DIR / "phase4_leading_xgboost_report.md").read_text(encoding="utf-8")))


# Temporal Robustness Experiment

## Decision

- Replacement promoted: **yes**
- Selected development candidate: `current_leading_row`
- Selected feature count: 38
- Selection used two rolling development folds; the final 15% user cohort remained untouched until this decision.

## Untouched Test Comparison

| Metric | Previous model | Candidate | Change |
|---|---:|---:|---:|
| Row ROC-AUC | 0.6607 | 0.6640 | +0.0034 |
| Row PR-AUC | 0.2585 | 0.2637 | +0.0052 |
| Latest-customer ROC-AUC | 0.6604 | 0.6707 | +0.0103 |
| Latest-customer PR-AUC | 0.2650 | 0.2750 | +0.0100 |

## Rolling Development Results

| Candidate | Features | Weighting | Mean ROC-AUC | Mean PR-AUC | Latest ROC-AUC | Latest PR-AUC |
|---|---:|---|---:|---:|---:|---:|
| `current_leading_row` | 38 | row | 0.7237 | 0.3706 | 0.7968 | 0.5030 |
| `regularized_leading_row` | 38 | row | 0.7236 | 0.3702 | 0.7965 | 0.5023 |
| `regularized_combined_row` | 45 | row | 0.7201 | 0.3673 | 0.7957 | 0.4992 |
| `regularized_drift_robust_user_recency` | 38 | user_recency | 0.7211 | 0.3659 | 0.7966 | 0.5012 |
| `regularized_combined_user_recency` | 45 | user_recency | 0.7178 | 0.3629 | 0.7963 | 0.5021 |
| `regularized_leading_user` | 38 | user | 0.7188 | 0.3611 | 0.7959 | 0.5009 |
| `regularized_combined_user` | 45 | user | 0.7177 | 0.3613 | 0.7960 | 0.5006 |
| `regularized_change_only_user_recency` | 23 | user_recency | 0.7060 | 0.3519 | 0.7807 | 0.4936 |

## Interpretation

The experiment targets lifecycle distribution shift rather than optimizing against the held-out test. User-balanced weighting makes each customer contribute equal total training weight; recency weighting emphasizes later development histories; drift-robust feature sets are selected using development folds only.


# Phase 4 Temporally Robust XGBoost Report

## Scope

Promoted model selected across two rolling user-lifecycle development folds, then evaluated once on the untouched final 15% user cohort.

## Test Metrics

- ROC-AUC: 0.6640
- PR-AUC: 0.2637
- Top 5% lift: 2.53x
- Top 10% lift: 2.18x
- Latest-customer ROC-AUC: 0.6707
- Latest-customer PR-AUC: 0.2750
- Precision at rolling-development threshold: 36.01%
- Recall at rolling-development threshold: 13.81%

## Timing Metric

- Correctly flagged users: 5,500 / 23,892
- Correctly flagged user rate: 23.02%
- Median lead time: 12.0 days

## Selected Design

- Candidate: `current_leading_row`
- Features: 38
- Weighting: `row`
- Regularization profile: `current`

See `reports/modeling/instacart/temporal_robustness_report.md` for all development candidates and the honest pre/post comparison.
